# RSNA Knee Abnormality Detection - Exploratory Data Analysis

This notebook performs Exploratory Data Analysis (EDA) for the RSNA Knee Abnormality Detection competition. We analyze:
1. **`train.csv`**: Study UIDs, free-text radiology reports, and sparse target labels.
2. **`train_series.csv`**: MRI series mapping, planes (Sagittal, Axial, Coronal), and fluid/fat suppression indicators.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_DIR = "data"
print("Libraries imported successfully.")

## 1. Loading the Datasets

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
series_df = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))

print(f"train.csv shape: {train_df.shape}")
print(f"train_series.csv shape: {series_df.shape}")

## 2. Profiling `train.csv` (Sparsity & Label Distribution)

Let's see how many rows are labeled versus unlabeled. Recall that only a very small subset of studies have clinician annotations.

In [ ]:
target_cols = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", 
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Check missing labels count
labeled_mask = train_df[target_cols].notnull().any(axis=1)
num_labeled = labeled_mask.sum()
num_unlabeled = len(train_df) - num_labeled

print(f"Number of labeled studies: {num_labeled} ({num_labeled/len(train_df)*100:.2f}%)")
print(f"Number of unlabeled studies: {num_unlabeled} ({num_unlabeled/len(train_df)*100:.2f}%)")

# Let's visualize this split
plt.figure(figsize=(6, 5))
sns.barplot(x=["Labeled (Clinician Ground Truth)", "Unlabeled (Report Only)"], y=[num_labeled, num_unlabeled], palette="viridis")
plt.title("Annotation Sparsity in train.csv")
plt.ylabel("Count of Studies")
plt.show()

### Target Labels Prevalence in the Labeled Subset
Let's inspect the positive and negative class counts for the 58 labeled studies.

In [ ]:
labeled_df = train_df[labeled_mask].copy()

# Melt the dataframe to make it easy to plot with seaborn
melted_df = pd.melt(labeled_df[target_cols])
melted_df["value"] = melted_df["value"].astype(str) # Convert labels to categorical strings for plotting

plt.figure(figsize=(14, 6))
sns.countplot(data=melted_df, x="variable", hue="value", palette="coolwarm")
plt.xticks(rotation=45)
plt.title("Distribution of Abnormalities in Labeled Studies (N = 58)")
plt.xlabel("Abnormality Class")
plt.ylabel("Count")
plt.legend(title="Status", labels=["Negative (0.0)", "Positive (1.0)"])
plt.tight_layout()
plt.show()

## 3. Profiling the Free-Text Reports

Since most of the dataset is unlabeled, we need to extract information from the `Report` column. Let's analyze report text lengths and view sample reports.

In [ ]:
# Report length statistics
train_df["Report_Length"] = train_df["Report"].fillna("").apply(len)
train_df["Word_Count"] = train_df["Report"].fillna("").apply(lambda x: len(x.split()))

print("=== Report Length Summary (Characters) ===")
print(train_df["Report_Length"].describe())
print("\n=== Report Word Count Summary ===")
print(train_df["Word_Count"].describe())

# Length distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train_df["Report_Length"], bins=50, kde=True, ax=axes[0], color="skyblue")
axes[0].set_title("Distribution of Report Lengths (Characters)")
axes[0].set_xlabel("Length (chars)")

sns.histplot(train_df["Word_Count"], bins=50, kde=True, ax=axes[1], color="salmon")
axes[1].set_title("Distribution of Report Word Counts")
axes[1].set_xlabel("Word Count")
plt.show()

### Sample Multilingual Reports
Let's print some reports of different lengths to see the variation in formatting and language.

In [ ]:
print("=== Sample 1 (Short/Dutch) ===")
print(train_df.iloc[1]["Report"])
print("-" * 50)

print("=== Sample 2 (Spanish) ===")
print(train_df.iloc[2]["Report"][:400])
print("...")
print("-" * 50)

print("=== Sample 3 (Longer Report) ===")
long_report = train_df[train_df["Report_Length"] > 800].iloc[0]["Report"]
print(long_report[:500])
print("...")

## 4. Profiling `train_series.csv` (Series Metadata)

Each knee MRI study consists of multiple series. Let's analyze how many series exist per study, which planes are most common, and how parameters like fat suppression are configured.

In [ ]:
# 1. Number of series per study
series_per_study = series_df.groupby("StudyInstanceUID").size()
print("=== Number of Series per Study ===")
print(series_per_study.describe())

plt.figure(figsize=(8, 4))
sns.countplot(x=series_per_study, palette="muted")
plt.title("Number of MRI Series per Study")
plt.xlabel("Number of Series")
plt.ylabel("Count of Studies")
plt.show()

### Anatomical Planes & MRI Contrast Parameters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plane Distribution
sns.countplot(data=series_df, x="Anatomical_Plane", order=series_df["Anatomical_Plane"].value_counts().index, ax=axes[0], palette="Set2")
axes[0].set_title("Distribution of Planes")
axes[0].set_xlabel("Plane")

# Fluid Sensitivity
sns.countplot(data=series_df, x="Fluid_Sensitive", ax=axes[1], palette="pastel")
axes[1].set_title("Fluid Sensitive")

# Fat Suppression
sns.countplot(data=series_df, x="Fat_Suppression", ax=axes[2], palette="pastel")
axes[2].set_title("Fat Suppression")

plt.tight_layout()
plt.show()

# Check correlation between Fluid Sensitive and Fat Suppression
cross_tab = pd.crosstab(series_df["Fluid_Sensitive"], series_df["Fat_Suppression"])
print("=== Cross-tabulation of Fluid Sensitive vs Fat Suppression ===")
print(cross_tab)

## 5. Main Takeaways from EDA

1. **Highly Unbalanced/Sparsely Annotated Targets:** Less than 1.5% of our dataset (58/4407) contains direct ground-truth classification labels. Designing a robust text keyword classifier (or pseudo-label generator) is critical to utilize the remaining 98.5% of studies.
2. **Multilingual Context:** Free-text reports are in multiple languages, demanding a multilingual keyword approach or multilingual LLM parser.
3. **Multi-Plane Input:** Studies typically contain 3 to 6 series across Axial, Coronal, and Sagittal views. A strong model needs to leverage these complementary planes or focus on the most diagnostic plane (like Sagittal for ACL/Meniscus).

# Phase 2: Report Parsing & Multilingual Label Extraction

In this section, we build a rule-based multilingual NLP parser to extract binary/soft target labels for the 12 abnormality categories from the free-text reports. 

We will:
1. Define a translation lexicon mapping medical terms for each abnormality to English, Spanish, Dutch, German, Turkish, Greek, and French keywords.
2. Implement a negation checker that accounts for prefix and suffix negation syntax.
3. Parse the unlabeled reports to generate pseudo-labels.
4. Validate the parser's performance against the 58 clinician-labeled ground-truth records.

In [19]:
# 1. Define the Multilingual Lexicon mapping target findings to keywords
LEXICON = {
    "ACL": {
        "keywords": [
            "acl", "lca", "cruzado anterior", "cruciate",
            "capraz", "kruisband", "ruptur", "ρήξη", "χιαστού"
        ]
    },
    "MCL": {
        "keywords": [
            "mcl", "lcm", "colateral medial", "collaterale",
            "yan bag", "innenband", "πλαγίου"
        ]
    },
    "Medial Meniscus": {
        "keywords": [
            "medial meniscus", "menisco medial", "menisco interno",
            "ic meniskus", "mediale meniscus", "innenmeniskus", "ménisque médial", "έσω μηνίσκου"
        ]
    },
    "Lateral Meniscus": {
        "keywords": [
            "lateral meniscus", "menisco lateral", "menisco externo",
            "dis meniskus", "laterale meniscus", "außenmeniskus", "ménisque latéral", "έξω μηνίσκου"
        ]
    },
    "Medial OA": {
        "keywords": [
            "medial oa", "artrosis medial", "artrosis femorotibial medial",
            "medial artroz", "mediale artrose", "mediale gonarthrose", "arthrose médiale"
        ]
    },
    "Lateral OA": {
        "keywords": [
            "lateral oa", "artrosis lateral", "artrosis femorotibial lateral",
            "dis artroz", "laterale artrose", "laterale gonarthrose", "arthrose latérale"
        ]
    },
    "PF OA": {
        "keywords": [
            "pf oa", "patellofemoral", "patelofemoral", "fémoro-patellaire",
            "femorotibial lateral", "patellofemorale", "επιγoνατιδoμηριαίας"
        ]
    },
    "Effusion": {
        "keywords": [
            "effusion", "derrame", "efüzyon", "fluid", "erguss",
            "effusie", "épanchement", "υγρού", "ύδραρθρο"
        ]
    },
    "Synovitis": {
        "keywords": [
            "synovitis", "sinovitis", "sinovit", "synoviite", "υμενίτιδα"
        ]
    },
    "Baker's": {
        "keywords": [
            "baker", "popliteal", "quiste de baker", "popliteal kist",
            "poplitealzyste", "kyste de baker", "κύστη baker"
        ]
    },
    "Contusion": {
        "keywords": [
            "contusion", "bruise", "bruising", "contusión",
            "kemik kontüzyonu", "contusie", "kontusion", "οίδημα"
        ]
    },
    "Fracture": {
        "keywords": [
            "fracture", "fractura", "kirik", "fractuur", "fraktur", "κατάγμα"
        ]
    }
}

print("Lexicon keywords defined for all 12 target classes.")

Lexicon keywords defined for all 12 target classes.


### 2. Implement the NLP Parsing Class

We compile a regex matching rule that checks for negation patterns (such as "no", "sin", "kein", "izlenmedi").

In [ ]:
import re
import unicodedata

# Compile negation terms (prefix and suffix)
PRE_NEGATIONS = r"\b(no|sin|not|negative for|without|free of|keine|kein|pas de|sans|normal|conservada|conservado|intacto|intacta)\b"
POST_NEGATIONS = r"\b(izlenmedi|saptanmadi|yok|not seen|absent|normaal|intact|normal)\b"

def clean_and_normalize(text):
    """Converts text to lowercase, strips accents, and cleans formatting."""
    if not isinstance(text, str):
        return ""
    # Strip accents for unicode compatibility (e.g. o -> o)
    nfkd_form = unicodedata.normalize('NFKD', text)
    text = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    return text.lower()

def check_negation(text, keyword, match_start, window=5):
    """
    Checks if a matching keyword is negated by scanning words
    within a specific sliding window before and after the match.
    """
    # Get words leading up to the keyword
    prefix_text = text[max(0, match_start - 60):match_start]
    prefix_words = re.findall(r'\w+', prefix_text)[-window:]
    if any(re.match(PRE_NEGATIONS, w) for w in prefix_words):
        return True
        
    # Get words following the keyword (critical for Turkish post-term negation)
    match_end = match_start + len(keyword)
    suffix_text = text[match_end:match_end + 60]
    suffix_words = re.findall(r'\w+', suffix_text)[:window]
    if any(re.match(POST_NEGATIONS, w) for w in suffix_words):
        return True
        
    return False

def parse_report(report_text):
    """
    Parses a report string and returns a dictionary of label predictions
    (1 for present, 0 for absent/not-mentioned/negated).
    """
    cleaned_report = clean_and_normalize(report_text)
    parsed_labels = {}
    
    for label, info in LEXICON.items():
        label_found = 0
        for kw in info["keywords"]:
            # Search for keyword matches
            for match in re.finditer(re.escape(kw), cleaned_report):
                match_start = match.start()
                # If keyword is present and NOT negated, label as positive
                if not check_negation(cleaned_report, kw, match_start):
                    label_found = 1
                    break
            if label_found == 1:
                break
        parsed_labels[label] = label_found
        
    return parsed_labels

### 3. Validation against Ground-Truth labels

Let's test the report parser on the 58 ground-truth studies to compute metrics (Precision, Recall, F1, and Accuracy) and verify how well the rule-based approach maps to clinical annotations.

In [21]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Filter the 58 studies that contain clinician-entered labels
target_cols = list(LEXICON.keys())
labeled_mask = train_df[target_cols].notnull().any(axis=1)
labeled_df = train_df[labeled_mask].copy()

gt_list = []
pred_list = []

for idx, row in labeled_df.iterrows():
    report_text = row["Report"]
    predictions = parse_report(report_text)
    
    # Align labels
    gt_labels = [int(row[col]) for col in target_cols]
    pred_labels = [predictions[col] for col in target_cols]
    
    gt_list.append(gt_labels)
    pred_list.append(pred_labels)

gt_array = np.array(gt_list)
pred_array = np.array(pred_list)

# Output classification metrics for each target class
print("=== Parser Validation Metrics (N=58 Ground-Truth Studies) ===\n")
for i, col in enumerate(target_cols):
    print(f"--- Class: {col} ---")
    print(f"Accuracy: {accuracy_score(gt_array[:, i], pred_array[:, i]):.3f}")
    print(f"F1 Score: {f1_score(gt_array[:, i], pred_array[:, i], zero_division=0):.3f}")
    print(classification_report(gt_array[:, i], pred_array[:, i], target_names=["Negative", "Positive"], zero_division=0))
    print("="*40)

### 4. Generate Pseudo-Labels for the Entire Dataset

Now we apply our validated parser to all 4,349 unlabeled records to generate targets for training the computer vision models.

In [22]:
pseudo_labeled_df = train_df.copy()
parsed_counts = 0

for idx, row in pseudo_labeled_df.iterrows():
    # If study does not have clinician labels, use parsed labels
    if pd.isnull(row[target_cols]).all():
        parsed_dict = parse_report(row["Report"])
        for col in target_cols:
            pseudo_labeled_df.at[idx, col] = parsed_dict[col]
        parsed_counts += 1

print(f"Successfully generated pseudo-labels for {parsed_counts} unlabeled records.")
print(f"Total pseudo-labeled training rows: {len(pseudo_labeled_df)}")

# Save the generated pseudo-labeled CSV file for training steps
output_csv = os.path.join(DATA_DIR, "train_pseudo_labeled.csv")
pseudo_labeled_df.to_csv(output_csv, index=False)
print(f"Saved pseudo-labeled dataset to: {output_csv}")

Successfully generated pseudo-labels for 4349 unlabeled records.
Total pseudo-labeled training rows: 4407
Saved pseudo-labeled dataset to: data\train_pseudo_labeled.csv
